# 10 - Uniform-Population Learning Curves

This notebook visualizes outputs from `experiments/uniform_population_learning.py`. It compares policies when true users are sampled from a near-uniform latent-trait population, while the inference model still starts from a Gaussian prior.

The main question is whether each policy becomes both accurate and confident over time:

- estimation error: how far posterior mean `mu` is from true `theta`
- uncertainty: how large posterior covariance remains
- dropout-aware progress: what happens when users stop answering

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError("Could not find project root containing src/ and experiments/.")


REPO_ROOT = find_project_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
RESULTS_DIR = REPO_ROOT / "experiments" / "results"

for path in [REPO_ROOT, NOTEBOOK_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

from _sweep_helpers import POLICY_STYLE, PALETTE, setup, style_ax

setup()
print(f"Repo root: {REPO_ROOT}")

## Load Result

Run the experiment first if no file is found:

```powershell
python -m experiments.uniform_population_learning --dim 6 --horizon 20 --users 500 --no-myopic
```

Leave `RUN_FILE = None` to load the newest uniform-learning result, or paste a JSON/`_curves.csv` filename from `experiments/results/`.

In [ ]:
RUN_FILE = None
# RUN_FILE = "20260429_175318_uniform_learning_dim2_h3_p10_n8.json"


def _uniform_learning_runs_newest_first() -> list[Path]:
    entries = []
    with os.scandir(RESULTS_DIR) as scan:
        for entry in scan:
            if entry.is_file() and entry.name.endswith(".json") and "uniform_learning" in entry.name:
                entries.append((entry.stat().st_mtime_ns, Path(entry.path)))
    return [path for _, path in sorted(entries, reverse=True)]


def list_uniform_learning_runs(limit: int = 12) -> list[Path]:
    files = _uniform_learning_runs_newest_first()
    if not files:
        print("No uniform-learning result files found.")
        return []
    for index, path in enumerate(files[:limit]):
        print(f"{index:>2}: {path.name}")
    return files


def resolve_uniform_learning_run(run_file: str | Path | None = None) -> Path:
    if run_file is None:
        files = list_uniform_learning_runs()
        if not files:
            raise FileNotFoundError("Run experiments.uniform_population_learning first.")
        return files[0]

    path = Path(run_file)
    if not path.is_absolute():
        path = RESULTS_DIR / path
    if path.name.endswith("_curves.csv"):
        path = path.with_name(path.name.removesuffix("_curves.csv") + ".json")
    if not path.exists():
        raise FileNotFoundError(f"Uniform-learning JSON not found: {path}")
    return path


def read_run_metadata(path: Path) -> dict:
    """Read config/policy metadata without parsing the potentially large curves array."""
    lines = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.lstrip().startswith('"curves"'):
                lines.append('  "curves": []\n')
                lines.append('}\n')
                break
            lines.append(line)
        else:
            return json.loads(path.read_text(encoding="utf-8"))
    return json.loads("".join(lines))


result_path = resolve_uniform_learning_run(RUN_FILE)
curve_path = result_path.with_name(f"{result_path.stem}_curves.csv")

data = read_run_metadata(result_path)
config = data["config"]
if curve_path.exists():
    curves = pd.read_csv(curve_path)
else:
    data = json.loads(result_path.read_text(encoding="utf-8"))
    curves = pd.DataFrame(data["curves"])
policies = list(data["policies"].keys())

print(f"Loaded: {result_path.name}")
print(f"Curves: {curve_path.name if curve_path.exists() else 'embedded JSON'}")
print(f"Policies: {', '.join(policies)}")
curves.head()

## Configuration

In [ ]:
config_df = pd.DataFrame(
    [{"parameter": key, "value": value} for key, value in config.items()]
)
config_df

## Final Policy Comparison

This table uses the final `carried` curve point. That means users who dropped out are still included, with their last posterior carried forward through the remaining horizon.

In [ ]:
final_carried = (
    curves[curves["mode"] == "carried"]
    .sort_values(["policy", "answered"])
    .groupby("policy", as_index=False)
    .tail(1)
)

policy_df = pd.DataFrame.from_dict(data["policies"], orient="index").reset_index(names="policy")
final_table = final_carried.merge(policy_df, on="policy", how="left")
final_table = final_table[
    [
        "policy",
        "dropout_rate",
        "mean_n_answered",
        "answered",
        "mean_l2_error",
        "rmse",
        "mean_d_error",
        "mean_trace",
        "sensitive_rate",
    ]
].sort_values("mean_l2_error")

final_table

## Estimation Error Over Time

`carried` is the main dropout-aware view: if a user drops out, their estimate stops improving and stays in the average.

In [ ]:
MODE = "carried"  # "carried" or "available"

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for policy in policies:
    style = POLICY_STYLE.get(policy, {})
    sub = curves[(curves["mode"] == MODE) & (curves["policy"] == policy)]
    label = style.get("label", policy)
    axes[0].plot(
        sub["answered"],
        sub["mean_l2_error"],
        color=style.get("color"),
        ls=style.get("ls", "-"),
        marker=style.get("marker", "o"),
        linewidth=1.8,
        markersize=4,
        label=label,
    )
    axes[1].plot(
        sub["answered"],
        sub["rmse"],
        color=style.get("color"),
        ls=style.get("ls", "-"),
        marker=style.get("marker", "o"),
        linewidth=1.8,
        markersize=4,
        label=label,
    )

axes[0].set_title("Mean L2 error")
axes[0].set_ylabel("distance to true theta")
axes[1].set_title("RMSE")

for ax in axes:
    ax.set_xlabel("Answered questions")
    style_ax(ax, grid_axis="y")

axes[1].legend(fontsize=8)
fig.suptitle(f"Estimation accuracy over time ({MODE})", y=1.03)
plt.tight_layout()
plt.show()

## Posterior Uncertainty Over Time

D-error is the geometric mean posterior variance. Trace is the sum of marginal posterior variances. Lower means the posterior is more certain.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for policy in policies:
    style = POLICY_STYLE.get(policy, {})
    sub = curves[(curves["mode"] == MODE) & (curves["policy"] == policy)]
    label = style.get("label", policy)
    axes[0].plot(
        sub["answered"],
        sub["mean_d_error"],
        color=style.get("color"),
        ls=style.get("ls", "-"),
        marker=style.get("marker", "o"),
        linewidth=1.8,
        markersize=4,
        label=label,
    )
    axes[1].plot(
        sub["answered"],
        sub["mean_trace"],
        color=style.get("color"),
        ls=style.get("ls", "-"),
        marker=style.get("marker", "o"),
        linewidth=1.8,
        markersize=4,
        label=label,
    )

axes[0].set_title("Mean D-error")
axes[0].set_ylabel("posterior uncertainty")
axes[1].set_title("Mean trace")

for ax in axes:
    ax.set_xlabel("Answered questions")
    style_ax(ax, grid_axis="y")

axes[1].legend(fontsize=8)
fig.suptitle(f"Posterior uncertainty over time ({MODE})", y=1.03)
plt.tight_layout()
plt.show()

## Trait-Wise Learning

The per-trait curves use posterior marginal variance. Stepwise reduction is the average drop in marginal variance from the previous answered-question count to the current one.

In [ ]:
TRAIT_POLICY = "surrogate_weighted" if "surrogate_weighted" in policies else policies[-1]


def trait_columns(prefix: str) -> list[str]:
    cols = [col for col in curves.columns if col.startswith(f"{prefix}_dim_")]
    return sorted(cols, key=lambda col: int(col.rsplit("_", 1)[-1]))


variance_cols = trait_columns("mean_variance")
step_reduction_cols = trait_columns("mean_variance_step_reduction")

if not step_reduction_cols:
    print("This run was generated before trait-wise step reductions were saved. Re-run experiments.uniform_population_learning to populate them.")
else:
    sub = curves[(curves["mode"] == MODE) & (curves["policy"] == TRAIT_POLICY)].sort_values("answered")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    colors = plt.cm.tab10(np.linspace(0, 1, len(step_reduction_cols)))

    for idx, (variance_col, reduction_col) in enumerate(zip(variance_cols, step_reduction_cols)):
        label = f"trait {idx + 1}"
        axes[0].plot(
            sub["answered"],
            sub[variance_col],
            color=colors[idx],
            marker="o",
            linewidth=1.6,
            markersize=3.5,
            label=label,
        )
        axes[1].plot(
            sub["answered"],
            sub[reduction_col],
            color=colors[idx],
            marker="o",
            linewidth=1.6,
            markersize=3.5,
            label=label,
        )

    axes[0].set_title("Mean marginal variance")
    axes[0].set_ylabel("posterior variance")
    axes[1].set_title("Stepwise variance reduction")
    axes[1].set_ylabel("variance drop since previous step")
    for ax in axes:
        ax.set_xlabel("Answered questions")
        style_ax(ax, grid_axis="y")
    axes[1].legend(fontsize=8, ncol=2)
    fig.suptitle(f"Trait-wise learning for {POLICY_STYLE.get(TRAIT_POLICY, {}).get('label', TRAIT_POLICY)} ({MODE})", y=1.03)
    plt.tight_layout()
    plt.show()

## Sensitive vs Other Trait Learning

For high-trait-tail runs, this averages the stepwise variance reduction over the configured sensitive axes and compares it with the remaining traits.

In [ ]:
step_reduction_cols = trait_columns("mean_variance_step_reduction")
sensitive_axes = config.get("sensitive_axes") or []
sensitive_axes = [axis for axis in sensitive_axes if 0 <= int(axis) < len(step_reduction_cols)]
other_axes = [axis for axis in range(len(step_reduction_cols)) if axis not in sensitive_axes]

if not step_reduction_cols:
    print("This run was generated before trait-wise step reductions were saved. Re-run experiments.uniform_population_learning to populate them.")
elif not sensitive_axes:
    print("No sensitive_axes configured for this run; this grouped view is most useful for high_trait_tail experiments.")
else:
    fig, axes = plt.subplots(1, len(policies), figsize=(3.3 * len(policies), 3.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, policy in zip(axes, policies):
        sub = curves[(curves["mode"] == MODE) & (curves["policy"] == policy)].sort_values("answered").copy()
        sensitive_cols = [step_reduction_cols[axis] for axis in sensitive_axes]
        other_cols = [step_reduction_cols[axis] for axis in other_axes]
        sub["sensitive_step_reduction"] = sub[sensitive_cols].mean(axis=1)
        if other_cols:
            sub["other_step_reduction"] = sub[other_cols].mean(axis=1)

        ax.plot(
            sub["answered"],
            sub["sensitive_step_reduction"],
            color=PALETTE["red"],
            marker="o",
            linewidth=1.6,
            markersize=3.5,
            label="sensitive traits",
        )
        if other_cols:
            ax.plot(
                sub["answered"],
                sub["other_step_reduction"],
                color=PALETTE["mist"],
                ls="--",
                marker="s",
                linewidth=1.6,
                markersize=3.5,
                label="other traits",
            )

        ax.set_title(POLICY_STYLE.get(policy, {}).get("label", policy))
        ax.set_xlabel("Answered questions")
        ax.set_ylabel("Stepwise variance reduction" if ax is axes[0] else "")
        style_ax(ax, grid_axis="y")
    axes[-1].legend(fontsize=8)
    fig.suptitle(f"Sensitive vs other trait learning ({MODE})", y=1.03)
    plt.tight_layout()
    plt.show()

## Accuracy Versus Confidence

This plot compares the final carried estimate error against final carried uncertainty. Lower-left is better: more accurate and more certain.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.8))

for _, row in final_table.iterrows():
    policy = row["policy"]
    style = POLICY_STYLE.get(policy, {})
    ax.scatter(
        row["mean_d_error"],
        row["mean_l2_error"],
        s=72,
        color=style.get("color"),
        marker=style.get("marker", "o"),
        edgecolor="white",
        linewidth=0.7,
        label=style.get("label", policy),
    )
    ax.annotate(
        style.get("label", policy),
        (row["mean_d_error"], row["mean_l2_error"]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_xlabel("Final mean D-error")
ax.set_ylabel("Final mean L2 error")
ax.set_title("Final accuracy vs uncertainty")
style_ax(ax, grid_axis="both")
plt.tight_layout()
plt.show()

## Carried Versus Available

`available` answers: among users who reached this answered-question count, how good is the estimate?

`carried` answers: across the full original population, how good is the estimate if dropout freezes learning?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for policy in policies:
    style = POLICY_STYLE.get(policy, {})
    for mode, alpha, linestyle_suffix in [("available", 0.45, ":"), ("carried", 1.0, "-")]:
        sub = curves[(curves["mode"] == mode) & (curves["policy"] == policy)]
        axes[0].plot(
            sub["answered"],
            sub["mean_l2_error"],
            color=style.get("color"),
            ls=linestyle_suffix,
            linewidth=1.8,
            alpha=alpha,
        )
        axes[1].plot(
            sub["answered"],
            sub["mean_d_error"],
            color=style.get("color"),
            ls=linestyle_suffix,
            linewidth=1.8,
            alpha=alpha,
        )

axes[0].set_title("Mean L2 error")
axes[0].set_ylabel("distance to true theta")
axes[1].set_title("Mean D-error")
axes[1].set_ylabel("posterior uncertainty")

for ax in axes:
    ax.set_xlabel("Answered questions")
    style_ax(ax, grid_axis="y")

axes[0].plot([], [], color=PALETTE["ink"], ls="-", label="carried")
axes[0].plot([], [], color=PALETTE["ink"], ls=":", label="available")
axes[0].legend(fontsize=8)
fig.suptitle("Dropout handling changes the denominator", y=1.03)
plt.tight_layout()
plt.show()

## Contributors Per Time Step

For `available`, later points can be based on fewer users because dropouts are removed from the denominator.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))

for policy in policies:
    style = POLICY_STYLE.get(policy, {})
    sub = curves[(curves["mode"] == "available") & (curves["policy"] == policy)]
    ax.plot(
        sub["answered"],
        sub["n"],
        color=style.get("color"),
        ls=style.get("ls", "-"),
        marker=style.get("marker", "o"),
        linewidth=1.7,
        markersize=4,
        label=style.get("label", policy),
    )

ax.set_xlabel("Answered questions")
ax.set_ylabel("episodes included")
ax.set_title("Available-curve sample size")
style_ax(ax, grid_axis="y")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Curve Data

Use this table for exact values or export.

In [ ]:
checkpoints = sorted(set([0, int(config["horizon"]) // 2, int(config["horizon"])]))
curves[curves["answered"].isin(checkpoints)].sort_values(
    ["mode", "answered", "policy"]
)